# Data Analysis - 2nd Part

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import matplotlib.image as mpimg

### Classification


In [3]:
clean_nhis = pd.read_csv('data/02-processed/nhis_2024_sample_adult_cleaned.csv')
df_ML=clean_nhis.copy()

df_ML['access_disruption_score'] = (
    (df_ML['delayed_medical_care'] == 1).astype(int)
    + (df_ML['usual_place_care'] != 'Has usual place of care').astype(int)
    + (df_ML['any_er_visit'] >= 1).astype(int)
    + (df_ML['doctor_gap_yr'] > 1).astype(int)
    + (df_ML['general_health'] <= 2).astype(int)
)
df_ML=df_ML.drop(columns=['government_rent_assistance'])
df_ML=df_ML.rename(columns={'AGEP_A':'age','POVRATTC_A':'poverty_ratio'})

In [4]:
df_ML['government_rent_assistance_cat']=clean_nhis['government_rent_assistance'].map({-1:'not applicable',0:'No',1:'yes'})


df_ML['health_insurance'] = clean_nhis['health_insurance'].map({
    'Uninsured': 0,
    'Insured': 1
})

X=df_ML.drop(columns=['HHX', 'WTFA_A', 'PSTRAT', 'PPSU','usual_place_care','general_health','delayed_medical_care', 'any_er_visit', 'doctor_gap_yr','access_disruption_score'])
y=df_ML['access_disruption_score']
w=df_ML["WTFA_A"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    stratify=y,
    random_state=42
)
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

In [ ]:

numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])
model = Pipeline([
    ('prep', preprocessor),
    ('svm',LogisticRegression(
    max_iter=5000))
])


In [ ]:
model.fit(
    X_train,
    y_train
    )
y_pred=model.predict(X_test)

In [ ]:

class_names = np.unique(np.concatenate((y_test, y_pred)))

conf_mat = confusion_matrix(y_test, y_pred)

mat=ConfusionMatrixDisplay(
    conf_mat,
    display_labels=class_names
).plot()

plt.xticks(rotation=45)
plt.title("Confusion Matrix (Predict Disruption Score)")

mat.figure_.savefig('results/classification_confusion_matrix.png', dpi=200)


report=classification_report(y_test, y_pred,output_dict=True)
rep_df = pd.DataFrame.from_dict(report)
rep_df.to_csv('results/classification_report.csv', index=True)


### Count Model: Healthcare Access Disruption Score


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display



analysis_data = clean_nhis.copy()
# Construct disruption score in the same way as classification analysis
analysis_data['access_disruption_score'] = (
    (analysis_data['delayed_medical_care'] == 1).astype(int)
    + (analysis_data['usual_place_care'] != 'Has usual place of care').astype(int)
    + (analysis_data['any_er_visit'] >= 1).astype(int)
    + (analysis_data['doctor_gap_yr'] > 1).astype(int)
    + (analysis_data['general_health'] <= 2).astype(int)
)

score_distribution = analysis_data['access_disruption_score'].value_counts().sort_index().to_frame('n')
score_distribution['percent'] = score_distribution['n'] / score_distribution['n'].sum() * 100

distribution_summary = pd.DataFrame({
    'mean': [analysis_data['access_disruption_score'].mean()],
    'variance': [analysis_data['access_disruption_score'].var()],
    'min': [analysis_data['access_disruption_score'].min()],
    'max': [analysis_data['access_disruption_score'].max()]
})
score_distribution.to_csv('results/access_disruption_score_distribution.csv',index=True)
distribution_summary.to_csv('results/access_disruption_score_summary.csv',index=False)


In [22]:
full_formula = '''access_disruption_score ~ transportation_barrier + housing_cost_trouble
+ C(housing_tenure, Treatment(reference="Owned or being bought"))
+ AGEP_A + POVRATTC_A
+ C(sex) + C(race_ethnicity) + C(education) + C(health_insurance)'''

full_model = smf.glm(
    formula=full_formula,
    data=analysis_data,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

renters = analysis_data[analysis_data['housing_tenure'] == 'Rented'].copy()
renter_formula = '''access_disruption_score ~ transportation_barrier + housing_cost_trouble
+ government_rent_assistance + AGEP_A + POVRATTC_A
+ C(sex) + C(race_ethnicity) + C(education) + C(health_insurance)'''

renter_model = smf.glm(
    formula=renter_formula,
    data=renters,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

def make_irr_table(result, model_name):
    conf = result.conf_int()
    table = pd.DataFrame({
        'model': model_name,
        'term': result.params.index,
        'coef': result.params.values,
        'IRR': np.exp(result.params.values),
        'CI_lower': np.exp(conf[0].values),
        'CI_upper': np.exp(conf[1].values),
        'p_value': result.pvalues.values
    })
    return table

full_results = make_irr_table(full_model, 'Full sample Poisson')
renter_results = make_irr_table(renter_model, 'Renter-only Poisson')
poisson_results = pd.concat([full_results, renter_results], ignore_index=True)

key_terms = [
    'transportation_barrier',
    'housing_cost_trouble',
    'C(housing_tenure, Treatment(reference="Owned or being bought"))[T.Rented]',
    'C(housing_tenure, Treatment(reference="Owned or being bought"))[T.Other arrangement]',
    'government_rent_assistance'
]
key_results = poisson_results[poisson_results['term'].isin(key_terms)].copy()

Path('results').mkdir(exist_ok=True)
poisson_results.to_csv('results/access_disruption_poisson_results.csv', index=False)
key_results.to_csv('results/access_disruption_poisson_key_results.csv', index=False)




In [ ]:
import matplotlib.pyplot as plt

key_results_for_plot = pd.read_csv('results/access_disruption_poisson_key_results.csv')
forest_data = key_results_for_plot[key_results_for_plot['model'] == 'Full sample Poisson'].copy()

label_map = {
    'transportation_barrier': 'Transportation barrier',
    'housing_cost_trouble': 'Housing cost trouble',
    'C(housing_tenure, Treatment(reference="Owned or being bought"))[T.Rented]': 'Rented vs owned',
    'C(housing_tenure, Treatment(reference="Owned or being bought"))[T.Other arrangement]': 'Other tenure vs owned'
}
forest_data['label'] = forest_data['term'].map(label_map)
forest_data = forest_data.dropna(subset=['label']).sort_values('IRR')

fig, ax = plt.subplots(figsize=(8, 4.8))
y_pos = np.arange(len(forest_data))
ax.errorbar(
    forest_data['IRR'],
    y_pos,
    xerr=[
        forest_data['IRR'] - forest_data['CI_lower'],
        forest_data['CI_upper'] - forest_data['IRR']
    ],
    fmt='o',
    color='#2F5D62',
    ecolor='#7B8D8E',
    elinewidth=2,
    capsize=4,
    markersize=7
)
ax.axvline(1, color='#555555', linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(forest_data['label'])
ax.set_xlabel('Incidence rate ratio (IRR) with 95% CI')
ax.set_title('Key Predictors of Healthcare Access Disruption')
ax.grid(axis='x', linestyle='--', alpha=0.35)
fig.tight_layout()
fig.savefig('results/access_disruption_irr_forest_plot.png', dpi=200)
plt.show()


### Additional Analyses Addressing Specific Aspects of the Outcomes

In [25]:
full_rhs = '''transportation_barrier + housing_cost_trouble
+ C(housing_tenure, Treatment(reference="Owned or being bought"))
+ AGEP_A + POVRATTC_A
+ C(sex) + C(race_ethnicity) + C(education) + C(health_insurance)'''

renter_rhs = '''transportation_barrier + housing_cost_trouble
+ government_rent_assistance + AGEP_A + POVRATTC_A
+ C(sex) + C(race_ethnicity) + C(education) + C(health_insurance)'''

def make_or_table(result, model_name):
    params = result.params
    conf   = result.conf_int()
    return (
        pd.DataFrame({
            'model':    model_name,
            'term':     params.index,
            'OR':       np.exp(params),
            'CI_lower': np.exp(conf[0]),
            'CI_upper': np.exp(conf[1]),
            'p_value':  result.pvalues
        })
        .query("term != 'Intercept'")
        .round({'OR': 4, 'CI_lower': 4, 'CI_upper': 4, 'p_value': 4})
    )


aspect_data = clean_nhis.copy()


aspect_data['has_usual_place'] = (
    aspect_data['usual_place_care'] == 'Has usual place of care'
).astype(int)
renters = aspect_data[aspect_data['housing_tenure'] == 'Rented'].copy()

delay_full = smf.glm(
    formula='delayed_medical_care ~ ' + full_rhs,
    data=aspect_data,
    family=sm.families.Binomial()
).fit(cov_type='HC1')

delay_renter = smf.glm(
    formula='delayed_medical_care ~ ' + renter_rhs,
    data=renters,
    family=sm.families.Binomial()
).fit(cov_type='HC1')

delay_results = pd.concat([
    make_or_table(delay_full,   'Full sample Logistic – delayed_medical_care'),
    make_or_table(delay_renter, 'Renter-only Logistic – delayed_medical_care')
], ignore_index=True)


usual_full = smf.glm(
    formula='has_usual_place ~ ' + full_rhs,
    data=aspect_data,
    family=sm.families.Binomial()
).fit(cov_type='HC1')

usual_renter = smf.glm(
    formula='has_usual_place ~ ' + renter_rhs,
    data=renters,
    family=sm.families.Binomial()
).fit(cov_type='HC1')

usual_results = pd.concat([
    make_or_table(usual_full,   'Full sample Logistic – has_usual_place'),
    make_or_table(usual_renter, 'Renter-only Logistic – has_usual_place')
], ignore_index=True)


doctor_full = smf.glm(
    formula='doctor_gap_yr ~ ' + full_rhs,
    data=aspect_data,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

doctor_renter = smf.glm(
    formula='doctor_gap_yr ~ ' + renter_rhs,
    data=renters,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

doctor_results = pd.concat([
    make_irr_table(doctor_full,   'Full sample Poisson – doctor_gap_yr'),
    make_irr_table(doctor_renter, 'Renter-only Poisson – doctor_gap_yr')
], ignore_index=True)

er_full = smf.glm(
    formula='any_er_visit ~ ' + full_rhs,
    data=aspect_data,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

er_renter = smf.glm(
    formula='any_er_visit ~ ' + renter_rhs,
    data=renters,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

er_results = pd.concat([
    make_irr_table(er_full,   'Full sample Poisson – any_er_visit'),
    make_irr_table(er_renter, 'Renter-only Poisson – any_er_visit')
], ignore_index=True)

health_full = smf.glm(
    formula='general_health ~ ' + full_rhs,
    data=aspect_data,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

health_renter = smf.glm(
    formula='general_health ~ ' + renter_rhs,
    data=renters,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

health_results = pd.concat([
    make_irr_table(health_full,   'Full sample Poisson – general_health'),
    make_irr_table(health_renter, 'Renter-only Poisson – general_health')
], ignore_index=True)


delay_results.to_csv('results/aspect_delayed_medical_care.csv', index=False)
usual_results.to_csv('results/aspect_usual_place_care.csv', index=False)
doctor_results.to_csv('results/aspect_doctor_gap_yr.csv', index=False)
er_results.to_csv('results/aspect_any_er_visit.csv', index=False)
health_results.to_csv('results/aspect_general_health.csv', index=False)

all_aspect_results = pd.concat([
    delay_results,
    usual_results,
    doctor_results,
    er_results,
    health_results
], ignore_index=True)

all_aspect_key_results = all_aspect_results[
    all_aspect_results['term'].isin(key_terms)
].copy()

all_aspect_results.to_csv('results/aspect_all_results.csv', index=False)
all_aspect_key_results.to_csv('results/aspect_key_results.csv', index=False)

## Poisson Prediction

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

rng = np.random.default_rng(108)

prediction_data = clean_nhis.copy()
prediction_data['access_disruption_score'] = (
    (prediction_data['delayed_medical_care'] == 1).astype(int)
    + (prediction_data['usual_place_care'] != 'Has usual place of care').astype(int)
    + (prediction_data['any_er_visit'] >= 1).astype(int)
    + (prediction_data['doctor_gap_yr'] > 1).astype(int)
    + (prediction_data['general_health'] <= 2).astype(int)
)

prediction_formula = '''access_disruption_score ~ transportation_barrier + housing_cost_trouble
+ C(housing_tenure, Treatment(reference="Owned or being bought"))
+ AGEP_A + POVRATTC_A
+ C(sex) + C(race_ethnicity) + C(education) + C(health_insurance)'''

train_mask = rng.random(len(prediction_data)) < 0.8
train_data = prediction_data.loc[train_mask].copy()
test_data = prediction_data.loc[~train_mask].copy()

prediction_model = smf.glm(
    formula=prediction_formula,
    data=train_data,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

test_data['predicted_access_disruption_score'] = prediction_model.predict(test_data)

prediction_performance = pd.DataFrame({
    'model': ['Poisson GLM', 'Train-mean baseline'],
    'MAE': [
        np.mean(np.abs(test_data['access_disruption_score'] - test_data['predicted_access_disruption_score'])),
        np.mean(np.abs(test_data['access_disruption_score'] - train_data['access_disruption_score'].mean()))
    ],
    'RMSE': [
        np.sqrt(np.mean((test_data['access_disruption_score'] - test_data['predicted_access_disruption_score']) ** 2)),
        np.sqrt(np.mean((test_data['access_disruption_score'] - train_data['access_disruption_score'].mean()) ** 2))
    ],
    'test_n': [len(test_data), len(test_data)]
})

full_prediction_model = smf.glm(
    formula=prediction_formula,
    data=prediction_data,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

def mode_value(series):
    return series.mode(dropna=True).iloc[0]

baseline_person = {
    'transportation_barrier': 0.0,
    'housing_cost_trouble': 0.0,
    'housing_tenure': 'Owned or being bought',
    'AGEP_A': prediction_data['AGEP_A'].median(),
    'POVRATTC_A': prediction_data['POVRATTC_A'].median(),
    'sex': mode_value(prediction_data['sex']),
    'race_ethnicity': mode_value(prediction_data['race_ethnicity']),
    'education': mode_value(prediction_data['education']),
    'health_insurance': mode_value(prediction_data['health_insurance'])
}

scenario_specs = [
    ('No transportation or housing barrier', 0.0, 0.0, 'Owned or being bought'),
    ('Transportation barrier only', 1.0, 0.0, 'Owned or being bought'),
    ('Housing cost trouble only', 0.0, 1.0, 'Owned or being bought'),
    ('Both transportation and housing barriers', 1.0, 1.0, 'Owned or being bought'),
    ('Renter, no listed barrier', 0.0, 0.0, 'Rented'),
    ('Renter with both barriers', 1.0, 1.0, 'Rented')
]

scenario_rows = []
for scenario, transportation, housing_trouble, tenure in scenario_specs:
    row = baseline_person.copy()
    row.update({
        'scenario': scenario,
        'transportation_barrier': transportation,
        'housing_cost_trouble': housing_trouble,
        'housing_tenure': tenure
    })
    scenario_rows.append(row)

scenario_predictions = pd.DataFrame(scenario_rows)
scenario_predictions['predicted_access_disruption_score'] = full_prediction_model.predict(scenario_predictions)
scenario_predictions['increase_vs_baseline'] = (
    scenario_predictions['predicted_access_disruption_score']
    - scenario_predictions.loc[0, 'predicted_access_disruption_score']
)
scenario_predictions['percent_increase_vs_baseline'] = (
    scenario_predictions['increase_vs_baseline']
    / scenario_predictions.loc[0, 'predicted_access_disruption_score']
    * 100
)

Path('results').mkdir(exist_ok=True)
prediction_performance.to_csv('results/access_disruption_prediction_performance.csv', index=False)
scenario_predictions[[
    'scenario', 'transportation_barrier', 'housing_cost_trouble', 'housing_tenure',
    'predicted_access_disruption_score', 'increase_vs_baseline', 'percent_increase_vs_baseline'
]].to_csv('results/access_disruption_scenario_predictions.csv', index=False)
test_data[['HHX', 'access_disruption_score', 'predicted_access_disruption_score']].to_csv(
    'results/access_disruption_test_predictions.csv', index=False
)



## Stratified by Race Ethnicity

In [ ]:

not_white = analysis_data[analysis_data['race_ethnicity'] != 'White non-Hispanic'].copy()

not_white_model = smf.glm(
    formula=full_formula,
    data=not_white,
    family=sm.families.Poisson()
).fit(cov_type='HC1')

nonwhite_renters=not_white[not_white['housing_tenure'] == 'Rented'].copy()
nonwhite_renter_model = smf.glm(
    formula=renter_formula,
    data=nonwhite_renters,
    family=sm.families.Poisson()
).fit(cov_type='HC1')



not_white_results = make_irr_table(not_white_model, 'Non-white subsample')
nonwhite_renter_results = make_irr_table(nonwhite_renter_model, 'Nonwhite_renter-only Poisson')
poisson_results   = pd.concat([not_white_results,nonwhite_renter_results], ignore_index=True)


key_results = poisson_results[poisson_results['term'].isin(key_terms)].copy()

Path('results').mkdir(exist_ok=True)
poisson_results.to_csv('results/access_non-white_subsample_poisson_results.csv',index=False)
key_results.to_csv('results/access_non-white_subsample_key_poisson_results.csv',index=False)


In [ ]:
scenario_predictions_for_plot = pd.read_csv('results/access_disruption_scenario_predictions.csv')
scenario_labels = [
    'No barrier',
    'Transport only',
    'Housing only',
    'Both barriers',
    'Renter only',
    'Renter + both'
]
scenario_predictions_for_plot['plot_label'] = scenario_labels

fig, ax = plt.subplots(figsize=(9, 5.2))
colors = ['#5B8C85', '#C77D4D', '#B85C5C', '#7B4E7A', '#4F6D9A', '#8B6F47']
bars = ax.bar(
    scenario_predictions_for_plot['plot_label'],
    scenario_predictions_for_plot['predicted_access_disruption_score'],
    color=colors,
    edgecolor='#2f2f2f',
    linewidth=0.7
)
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.025,
        f'{height:.2f}',
        ha='center',
        va='bottom',
        fontsize=10
    )
ax.set_ylabel('Predicted access disruption score')
ax.set_xlabel('Scenario')
ax.set_title('Predicted Healthcare Access Disruption by Barrier Scenario')
ax.set_ylim(0, max(scenario_predictions_for_plot['predicted_access_disruption_score']) * 1.22)
ax.tick_params(axis='x', rotation=20)
for label in ax.get_xticklabels():
    label.set_ha('right')
ax.grid(axis='y', linestyle='--', alpha=0.35)
fig.tight_layout()
fig.savefig('results/access_disruption_scenario_predictions.png', dpi=200)
plt.show()
